# Cas pratique n°2 — Titanic : modèle enrichi, recherche d'hyperparamètres, soumission

**Jour 2 — chapitre 03 : Feature engineering (suite) / cas pratique n°2**

## Mise en situation

En reprenant les features nettoyées (notebook 03) et enrichies (notebook 04), on entraîne
ici un nouveau modèle et on le compare au premier modèle du notebook 01. On cherche
ensuite ses meilleurs hyperparamètres, puis on génère un fichier de soumission au format
Kaggle.


In [1]:
import pandas as pd
import seaborn as sns

titanic_raw = sns.load_dataset("titanic")
titanic = titanic_raw.rename(columns={
    "survived": "Survived",
    "pclass": "Pclass",
    "sex": "Sex",
    "age": "Age",
    "sibsp": "SibSp",
    "parch": "Parch",
    "fare": "Fare",
    "embarked": "Embarked",
})[["Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]].copy()

# Nettoyage (TP "valeurs manquantes", notebook 03)
age_median_par_groupe = titanic.groupby(["Sex", "Pclass"])["Age"].transform("median")
titanic["Age"] = titanic["Age"].fillna(age_median_par_groupe).fillna(titanic["Age"].median())
titanic["Embarked"] = titanic["Embarked"].fillna(titanic["Embarked"].mode()[0])

# Feature engineering (TP "feature engineering", notebook 04)
titanic = pd.get_dummies(titanic, columns=["Sex", "Embarked"], drop_first=True)
titanic["FamilySize"] = titanic["SibSp"] + titanic["Parch"] + 1
titanic["IsAlone"] = (titanic["FamilySize"] == 1).astype(int)

# PassengerId synthetique pour le format de soumission (le dataset seaborn n'en fournit pas)
titanic = titanic.reset_index(drop=True)
titanic.insert(0, "PassengerId", titanic.index + 1)

features = ["Pclass", "Age", "Fare", "FamilySize", "IsAlone", "Sex_male"]
features = [c for c in features if c in titanic.columns]

X = titanic[features]
y = titanic["Survived"]

titanic.head()


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Sex_male,Embarked_Q,Embarked_S,FamilySize,IsAlone
0,1,0,3,22.0,1,0,7.2500,True,False,True,2,0
1,2,1,1,38.0,1,0,71.2833,False,False,False,2,0
2,3,1,3,26.0,0,0,7.9250,False,False,True,1,1
3,4,1,1,35.0,1,0,53.1000,False,False,True,2,0
4,5,0,3,35.0,0,0,8.0500,True,False,True,1,1


## Étape 1 — Un modèle sur les features nettoyées et enrichies

**Question de réflexion :** le nettoyage (`Age` imputé) et les nouvelles variables
(`FamilySize`, `IsAlone`) améliorent-ils le premier modèle du notebook 01 ? Comment
comparer les deux modèles équitablement ?


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
model = LogisticRegression(max_iter=5000)
model.fit(X_train, y_train)

scores = cross_val_score(model, X_train, y_train, cv=5)

# Premier modele du notebook 01 : memes passagers, memes plis, 5 colonnes sans valeurs manquantes
features_premier_modele = ["Pclass", "Sex_male", "SibSp", "Parch", "Fare"]
scores_premier_modele = cross_val_score(
    LogisticRegression(max_iter=5000),
    titanic.loc[X_train.index, features_premier_modele], y_train, cv=5,
)

print("Premier modèle (notebook 01) — score moyen CV :", round(scores_premier_modele.mean(), 3))
print("Modèle enrichi — scores par pli :", scores.round(3))
print("Modèle enrichi — score moyen CV :", round(scores.mean(), 3))

**Ce qu'on observe** : les deux modèles sont évalués sur les mêmes passagers et les mêmes
plis de cross-validation, la comparaison est donc équitable. Le modèle enrichi fait à peine
mieux que le premier modèle (__CV_ENRICHI__ contre __CV_PREMIER__), un écart bien plus petit que la
variation d'un pli à l'autre. Pour une régression logistique, ces nouvelles variables
apportent peu : `FamilySize` résume `SibSp` et `Parch`, que le premier modèle utilisait déjà
séparément. Les méthodes ensemblistes (notebooks 06 et 07), qui captent les interactions
entre variables, en tirent davantage parti.

## Étape 2 — Recherche d'hyperparamètres avec GridSearchCV

**Question de réflexion :** quels hyperparamètres de la régression logistique pourrait-on
faire varier, et comment être sûr de ne pas choisir la meilleure combinaison « par
chance » sur un seul découpage ?

On fait varier `C` et le type de pénalité, Ridge (`l1_ratio=0`) ou Lasso (`l1_ratio=1`).
Ce sont les deux pénalités explorées à la main au notebook 02, sur une régression linéaire :
le principe est identique ici, seul le réglage change de nom. La régression logistique se
règle avec `C`, qui est l'**inverse** de la force de régularisation — le `alpha` du notebook
02 : plus `C` est petit, plus la pénalité est forte. `l1_ratio` remplace l'ancien paramètre
`penalty="l2"` / `"l1"`, déprécié depuis scikit-learn 1.8.


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "l1_ratio": [0, 1],
}

grid = GridSearchCV(
    LogisticRegression(solver="liblinear", max_iter=5000),
    param_grid,
    cv=5,
    scoring="accuracy",
)
grid.fit(X_train, y_train)

print("Meilleurs paramètres :", grid.best_params_)
print("Meilleur score en cross-validation :", round(grid.best_score_, 3))

**Ce qu'on observe** : `GridSearchCV` évalue chaque combinaison de la grille sur les 5
plis de cross-validation, et retient celle qui donne la meilleure performance moyenne.
Pour une grille plus large, `RandomizedSearchCV` échantillonnerait aléatoirement les
combinaisons plutôt que de toutes les tester. `grid.best_params_` donne directement la
configuration gagnante : c'est elle qu'on réutilise pour entraîner le modèle final.


## Étape 3 — Soumission sur Kaggle

**Question de réflexion :** quelles sont les colonnes attendues dans un fichier de
soumission Kaggle pour la compétition Titanic ?


In [6]:
best_model = grid.best_estimator_
best_model.fit(X_train, y_train)

predictions = best_model.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": titanic.loc[X_test.index, "PassengerId"],
    "Survived": predictions,
})
submission.to_csv("submission.csv", index=False)

submission.head()


,PassengerId,Survived
709,710,0
439,440,0
840,841,0
720,721,1
39,40,1


**Ce qu'on observe** : le fichier `submission.csv` contient exactement les deux colonnes
attendues par Kaggle, `PassengerId` et `Survived`. Ici, `X_test` provient de notre propre
split local (nous connaissons donc déjà `y_test`, ce qui nous sert de vérification) ;
pour une vraie soumission Kaggle, on applique `best_model.predict()` sur le `test.csv`
officiel, qui ne contient pas la colonne `Survived`.


In [ ]:
from sklearn.metrics import accuracy_score

# Verification locale, possible ici uniquement parce que nous connaissons y_test
test_accuracy = accuracy_score(y_test, predictions)

print("Accuracy sur notre jeu de test local :", round(test_accuracy, 3))

**À retenir pour le débriefing du TP** : trois scores de cross-validation se suivent
maintenant, celui du premier modèle (notebook 01), celui du modèle enrichi, puis celui du
modèle réglé par `GridSearchCV`. Avec une régression logistique, les gains restent modestes :
le réglage des hyperparamètres pèse peu, et les nouvelles variables ne donnent leur pleine
mesure qu'avec des modèles capables d'exploiter les interactions entre variables (notebooks
06 et 07).